# 01b · Linear models and linear dimension reduction (two-stage, closed form)

The improved linear baselines and the dimension-reduction methods. All of them predict with a single coefficient
vector, so each is fit in closed form from one pass over the window's training data (n, Σx, X'X, X'y and per-class
sums, accumulated on the GPU). That makes it cheap to run every family with a hyperparameter grid on every window.

| Family | What it is | Grid (chosen per window on validation) |
|---|---|---|
| OLS | least squares, as the paper's Linear but fit exactly | none |
| Ridge | L2 penalty λ × average feature variance | λ ∈ {0.001, 0.01, 0.1, 1, 10} |
| Ridge-rank | ridge on each day's return rank, rescaled to return units | same λ |
| PCR | regression on the top-K principal components of the numeric features, plus industry dummies | K ∈ {5, 10, 20, 40, 80} |
| PLS | partial least squares (SIMPLS) | K ∈ {1, 2, 3, 5, 8, 12, 20} |
| LDA | regression on the top-m Linear Discriminant Analysis directions separating each day's return deciles | m ∈ {1, 3, 5, 9}, shrinkage α ∈ {0.05, 0.2, 0.5} |
| LDA-top | Fisher discriminant for "top decile vs the rest", aimed at the long-only portfolio | α ∈ {0.01, 0.05, 0.2, 0.5} |

Everything (PCA loadings, LDA directions, coefficients) is fit on the training set only. The grid is chosen on
the validation set by **mean daily rank IC**, and the test quarter is never used for any choice. Every setting
tried is saved in `linear_grid.csv`, so the full grid can be reported (see PAPER_ISSUES.md on multiple testing).
The estimators are checked against scikit-learn in `checks/test_linear.py`.

In [1]:
from pathlib import Path

import pandas as pd

from mlm.data import panel_features, rolling_windows
from mlm.linear import GRIDS, run_linear

PANEL_DIR = Path("D:/ML_Markowitz_data/panel_cn")
PRED_DIR = Path("D:/ML_Markowitz_data/preds/cn")
FIRST_TEST, LAST_TEST = "2014-01-01", "2023-12-31"   # all 40 windows
FAMILIES = list(GRIDS)
CRITERION = "ic"
windows = rolling_windows(FIRST_TEST, LAST_TEST)
print(f"{len(windows)} windows, {len(panel_features(PANEL_DIR))} features, families: {FAMILIES}")

40 windows, 251 features, families: ['OLS', 'Ridge', 'Ridge-rank', 'PCR', 'PLS', 'LDA', 'LDA-top']


In [2]:
grid = run_linear(PANEL_DIR, windows, PRED_DIR, FAMILIES, CRITERION)

2014Q1: 1,472,279 train rows, stats 6s, total 14s | OLS {} ic=0.0950, Ridge {'lam': 0.1} ic=0.0988, Ridge-rank {'lam': 0.1} ic=0.1056, PCR {'k': 80} ic=0.0943, PLS {'k': 5} ic=0.1006, LDA {'m': 3, 'alpha': 0.05} ic=0.1046, LDA-top {'alpha': 0.05} ic=0.0289


2014Q2: 1,520,359 train rows, stats 6s, total 14s | OLS {} ic=0.0948, Ridge {'lam': 0.1} ic=0.0995, Ridge-rank {'lam': 0.1} ic=0.1034, PCR {'k': 80} ic=0.0963, PLS {'k': 8} ic=0.1008, LDA {'m': 3, 'alpha': 0.05} ic=0.1015, LDA-top {'alpha': 0.05} ic=0.0394


2014Q3: 1,579,898 train rows, stats 6s, total 13s | OLS {} ic=0.0950, Ridge {'lam': 0.1} ic=0.0991, Ridge-rank {'lam': 0.1} ic=0.1026, PCR {'k': 80} ic=0.0966, PLS {'k': 8} ic=0.1005, LDA {'m': 3, 'alpha': 0.05} ic=0.1014, LDA-top {'alpha': 0.2} ic=0.0277


2014Q4: 1,619,815 train rows, stats 6s, total 13s | OLS {} ic=0.0936, Ridge {'lam': 0.1} ic=0.0957, Ridge-rank {'lam': 0.01} ic=0.0976, PCR {'k': 80} ic=0.0936, PLS {'k': 8} ic=0.0963, LDA {'m': 3, 'alpha': 0.05} ic=0.0962, LDA-top {'alpha': 0.05} ic=0.0444


2015Q1: 1,650,427 train rows, stats 6s, total 13s | OLS {} ic=0.0855, Ridge {'lam': 0.1} ic=0.0874, Ridge-rank {'lam': 0.01} ic=0.0928, PCR {'k': 80} ic=0.0844, PLS {'k': 5} ic=0.0872, LDA {'m': 3, 'alpha': 0.05} ic=0.0878, LDA-top {'alpha': 0.2} ic=0.0552


...

2022Q4: 2,902,066 train rows, stats 11s, total 26s | OLS {} ic=0.0617, Ridge {'lam': 10.0} ic=0.0725, Ridge-rank {'lam': 0.001} ic=0.0792, PCR {'k': 20} ic=0.0672, PLS {'k': 2} ic=0.0813, LDA {'m': 1, 'alpha': 0.05} ic=0.0689, LDA-top {'alpha': 0.05} ic=0.0853


2023Q1: 2,950,029 train rows, stats 11s, total 27s | OLS {} ic=0.0732, Ridge {'lam': 1.0} ic=0.0842, Ridge-rank {'lam': 0.1} ic=0.0899, PCR {'k': 40} ic=0.0814, PLS {'k': 2} ic=0.0835, LDA {'m': 1, 'alpha': 0.05} ic=0.0789, LDA-top {'alpha': 0.05} ic=0.0853


2023Q2: 3,011,628 train rows, stats 12s, total 28s | OLS {} ic=0.0648, Ridge {'lam': 1.0} ic=0.0766, Ridge-rank {'lam': 0.1} ic=0.0875, PCR {'k': 40} ic=0.0729, PLS {'k': 2} ic=0.0764, LDA {'m': 1, 'alpha': 0.05} ic=0.0721, LDA-top {'alpha': 0.05} ic=0.0878


2023Q3: 3,099,930 train rows, stats 12s, total 28s | OLS {} ic=0.0506, Ridge {'lam': 1.0} ic=0.0606, Ridge-rank {'lam': 0.01} ic=0.0720, PCR {'k': 40} ic=0.0539, PLS {'k': 3} ic=0.0632, LDA {'m': 1, 'alpha': 0.05} ic=0.0587, LDA-top {'alpha': 0.01} ic=0.0765


2023Q4: 3,152,443 train rows, stats 13s, total 29s | OLS {} ic=0.0577, Ridge {'lam': 10.0} ic=0.0817, Ridge-rank {'lam': 1.0} ic=0.0882, PCR {'k': 20} ic=0.0683, PLS {'k': 2} ic=0.0849, LDA {'m': 1, 'alpha': 0.05} ic=0.0750, LDA-top {'alpha': 0.01} ic=0.0940


## Chosen settings and validation scores

In [3]:
chosen = grid[grid["chosen"] == True]
print(chosen.groupby("family")["params"].agg(lambda s: s.value_counts().head(3).to_dict()))
chosen.groupby("family")[["ic", "top", "mse"]].mean().sort_values("ic", ascending=False)

family
LDA           {'{'m': 3, 'alpha': 0.05}': 18, '{'m': 1, 'alp...
LDA-top       {'{'alpha': 0.05}': 13, '{'alpha': 0.2}': 12, ...
OLS                                                  {'{}': 40}
PCR           {'{'k': 80}': 19, '{'k': 40}': 15, '{'k': 20}'...
PLS             {'{'k': 2}': 16, '{'k': 20}': 7, '{'k': 5}': 6}
Ridge         {'{'lam': 1.0}': 14, '{'lam': 0.1}': 12, '{'la...
Ridge-rank    {'{'lam': 0.01}': 13, '{'lam': 0.1}': 12, '{'l...
Name: params, dtype: object


,ic,top,mse
family,,,
Ridge-rank,0.080070,0.001153,0.000624
LDA,0.076640,0.001192,0.000624
PLS,0.075466,0.001267,0.000624
Ridge,0.074083,0.001251,0.000624
PCR,0.069174,0.001102,0.000624
OLS,0.068124,0.001223,0.000624
LDA-top,0.059213,0.000516,0.000624
